# 🎙️ Model Server trên Colab T4 — emotion-chatbot

Notebook của **Quyên (ML)** — chạy CafeBERT (emotion) + PhoWhisper (speech) + Qwen3-4B (llm)
làm server FastAPI `:8001`, mở tunnel ngrok cho backend của Hiển gọi qua HTTP (mode `remote`).

Pipeline chi tiết: `server_app.py` → `backend/modules/{emotion,speech,llm}.py`.

Đọc thêm: `colab/README.md` (hợp đồng HTTP, quy tắc).

## 1. Cấu hình Runtime

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU` (bắt buộc — Qwen3-4B fp16 cần ~8GB VRAM).

Sau đó chạy tuần tự các ô bên dưới.

In [ ]:
# Clone repo + cài phụ thuộc
# ⚠️ Thay <URL-repo-emotion-chatbot> bằng URL repo nhóm của bạn
# ⚠️ Clone nhánh ML của Quyên (chưa merge vào main):
#   Sau khi PR merge, đổi lại thành repo NhienTatHien + nhánh main.
!git clone -b feat/quiniu-ml-modules https://github.com/quyen0723/emotion-chatbot.git /content/emotion-chatbot
%cd /content/emotion-chatbot
!pip install -q -r colab/requirements.txt -r backend/requirements-ml.txt pyngrok
# Phụ thuộc thêm cần thiết cho transformers 5.x (chỉ ra qua test local):
#  - sentencepiece + protobuf : tokenizer CafeBERT (sentencepiece.bpe.model)
#  - accelerate               : device_map của Qwen
!pip install -q sentencepiece protobuf accelerate

## 2. Secrets

Đặt trong **Colab Secrets** (biểu tượng 🔑 bên trái panel) — KHÔNG dán token vào ô:
- `MODEL_SERVER_TOKEN` — token gọi server (cùng giá trị với `.env` của backend)
- `NGROK_AUTHTOKEN` — token ngrok (đăng ký miễn phí dashboard ngrok → Auth Token)

In [ ]:
# Đọc secrets vào env
import os
from google.colab import userdata
os.environ["MODEL_SERVER_TOKEN"] = userdata.get("MODEL_SERVER_TOKEN")
os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")

# Chọn model LLM chạy trên T4 (đúng phương án của Thầy: Qwen3-4B).
# Env chính thức theo quy ước repo (reference_modules/llm.py): LLM_MODEL_ID.
# fp16 trên T4 ≈ 8GB VRAM; chế độ thinking được tắt sẵn trong modules/llm.py.
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen3-4B"
print("LLM_MODEL_ID =", os.environ["LLM_MODEL_ID"])

## 3. Khởi động model server

`SERVER_USE_MOCKS=0` (mặc định) → dùng `backend/modules/*` (model thật).
Lần đầu tạo process sẽ tải: CafeBERT (~500MB) + PhoWhisper-base (~1GB) + Qwen3-4B (~8GB) ≈ 10 phút.

In [ ]:
import subprocess
import os, signal

# Tắt server cũ nếu cell được chạy lại (rất hay xảy ra khi iterate)
subprocess.run("pkill -f 'uvicorn server_app' || true", shell=True)

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "server_app:app", "--host", "0.0.0.0", "--port", "8001"],
    cwd="/content/emotion-chatbot/colab",
    stdout=open("/content/server.log", "w"),
    stderr=subprocess.STDOUT,
)
print("server PID:", server.pid)

In [ ]:
# Chờ server có model sẵn (poll /health local). Ước tính: 2-10 phút lần đầu.
import time, urllib.request
for i in range(120):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=2)
        print("SERVER UP:", r.read().decode()); break
    except Exception:
        time.sleep(5)
else:
    print("Chưa sống sau 10 phút — xem /content/server.log")
!curl -s http://127.0.0.1:8001/health

## 4. Kiểm tra từng endpoint (trước khi mở tunnel)

In [ ]:
# Test /emotion (CafeBERT) — cần token nếu MODEL_SERVER_TOKEN được set
import json
import os
import urllib.request

TOKEN = os.environ["MODEL_SERVER_TOKEN"]
def post_json(path, payload):
    req = urllib.request.Request(
        f"http://127.0.0.1:8001{path}",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json", "X-Model-Server-Token": TOKEN},
    )
    return urllib.request.urlopen(req, timeout=120).read().decode()

print("emotion:", post_json("/emotion", {"text": "mình buồn quá"}))
# (Lần đầu mỗi endpoint sẽ load model — chậm là bình thường)

In [ ]:
# Test /generate (Qwen3-4B) — lần đầu load ~8GB, chờ 3-6 phút
import os, json, urllib.request

def post_json2(path, payload):
    data = json.dumps(payload).encode()
    req = urllib.request.Request(
        f"http://127.0.0.1:8001{path}",
        data=data,
        headers={"Content-Type": "application/json", "X-Model-Server-Token": os.environ["MODEL_SERVER_TOKEN"]},
    )
    return urllib.request.urlopen(req, timeout=600).read().decode()

print("generate:", post_json2("/generate", {
    "text": "mình buồn quá",
    "emotion": {"label": "Sadness"},
    "arousal": "medium",
    "history": [],
    "goal": "xoa_diu",
}))

## 5. Mở tunnel ngrok (static domain)

Đăng ký static domain miễn phí tại dashboard ngrok → **Domains** (vd `abc.ngrok-free.app`),
đổi `<ten-mien-tinh>.ngrok-free.app` bên dưới. Static domain quan trọng vì backend
cấu hình URL một lần; nếu không dùng static domain, URL ngrok mỗi lần chạy khác nhau.

In [ ]:
!ngrok config add-authtoken $NGROK_AUTHTOKEN
!pkill ngrok 2>/dev/null || true
# ⚠️ Đổi domain của bạn:
!nohup ngrok http --url=ten-mien-tinh.ngrok-free.app 8001 > /content/ngrok.log 2>&1 &
!sleep 3 && tail -3 /content/ngrok.log

In [ ]:
# Verify tunnel từ bên ngoài
!curl -s https://ten-mien-tinh.ngrok-free.app/health

## 6. Đưa `.env` cho backend của Hiển

Gửi cho Hiển bộ giá trị sau (chỉ qua kênh nhóm — không commit):
```env
EMOTION_MODE=remote
SPEECH_MODE=remote
LLM_MODE=remote
MODEL_SERVER_URL=https://ten-mien-tinh.ngrok-free.app
MODEL_SERVER_TOKEN=<MODEL_SERVER_TOKEN trên Colab Secrets>
# SAFE MUSIC giữ nguyên: SAFETY_MODE/MUSIC_MODE mock hoặc local
```
Backend restart với `.env` này là FE sẽ gọi model thật qua Colab.

⚠️ Khi Colab ngắt: backend tự dùng fallback (cảm xúc Other + Gemini/canned) — không crash.